In [2]:
import os
import datetime

LOG_DIR = r"C:\Users\bhanu\Downloads\PythonCode\banking_project\logs"
os.makedirs(LOG_DIR, exist_ok=True)

def log(message, level="INFO"):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] [{level}] {message}"
    print(log_line)
    
    log_file = os.path.join(LOG_DIR, f"pipeline_{datetime.date.today()}.log")
    with open(log_file, 'a') as f:
        f.write(log_line + "\n")

log("Logger ready.")

[2026-04-11 21:59:08] [INFO] Logger ready.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os

# --- Config ---
DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": 1234,   # change this
    "database": "banking_project"
}

# --- Connection ---
def get_engine():
    cfg = DB_CONFIG
    url = f"mysql+pymysql://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['database']}"
    return create_engine(url)

# --- Load CSV to staging table ---
def load_csv_to_staging(filepath, table_name, engine):
    print(f"Loading {filepath} into {table_name}...")
    
    chunks = pd.read_csv(filepath, chunksize=10000)
    
    first_chunk = True
    total_rows = 0
    
    for chunk in chunks:
        chunk.columns = chunk.columns.str.lower().str.replace(' ', '_')
        
        if first_chunk:
            chunk.to_sql(table_name, engine, if_exists='replace', index=False)
            first_chunk = False
        else:
            chunk.to_sql(table_name, engine, if_exists='append', index=False)
        
        total_rows += len(chunk)
        print(f"  Loaded {total_rows} rows so far...")
    
    print(f"Done. Total rows loaded into {table_name}: {total_rows}\n")

# --- Main ---
engine = get_engine()

# Change this to your actual data folder path
base_path = r"C:\Users\bhanu\Downloads\PythonCode\banking_project\data"

files = {
    "stg_applications": "application_train.csv",
    "stg_installments": "installments_payments.csv"
}

for table, filename in files.items():
    filepath = os.path.join(base_path, filename)
    if os.path.exists(filepath):
        load_csv_to_staging(filepath, table, engine)
    else:
        print(f"File not found: {filepath} — skipping")

print("Ingestion complete.")

####  Full Pipeline Runner

In [3]:
log("Pipeline automation ready.")
log("All tables already built and verified.")
log("Pipeline will re-run ingestion + transformations when new data arrives.")

[2026-04-11 21:59:12] [INFO] Pipeline automation ready.
[2026-04-11 21:59:12] [INFO] All tables already built and verified.
[2026-04-11 21:59:12] [INFO] Pipeline will re-run ingestion + transformations when new data arrives.


In [ ]:
def run_pipeline():
    start = datetime.datetime.now()
    log("====== PIPELINE STARTED ======")

    try:
        # Ingest
        base_path = r"C:\Users\bhanu\Downloads\PythonCode\banking_project\data"
        files = {
            "stg_applications": "application_train.csv",
            "stg_installments": "installments_payments.csv"
        }
        for table, filename in files.items():
            filepath = os.path.join(base_path, filename)
            load_csv_to_staging(filepath, table, engine)

        # Transform
        build_dim_customer(engine)
        build_dim_loan(engine)
        build_dim_payment(engine)
        build_fact_loan_performance(engine)

        duration = round((datetime.datetime.now() - start).total_seconds() / 60, 2)
        log(f"====== PIPELINE COMPLETED in {duration} mins ======")

    except Exception as e:
        log(f"PIPELINE FAILED: {str(e)}", level="ERROR")
        raise

run_pipeline()